In [1]:
# ============================================================
# CELL 1: Mount Drive + Config + Install
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, torch

class Config:
    PROJECT_ROOT    = '/content/drive/MyDrive/FinDocVQA'
    DATA_DIR        = os.path.join(PROJECT_ROOT, 'data')
    DOCVQA_DIR      = os.path.join(DATA_DIR, 'docvqa')
    CHARTQA_DIR     = os.path.join(DATA_DIR, 'chartqa')
    MODELS_DIR      = os.path.join(PROJECT_ROOT, 'models')
    OUTPUTS_DIR     = os.path.join(PROJECT_ROOT, 'outputs')
    PIX2STRUCT_ID   = 'google/pix2struct-docvqa-base'
    SEED = 42

cfg = Config()
print(f"✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Mounted at /content/drive
✓ GPU: Tesla T4


In [2]:
# ============================================================
# CELL 2: Install deps + Load datasets from Drive
# ============================================================

%%capture
!pip install transformers accelerate datasets sentencepiece protobuf

# ---
from datasets import load_from_disk
import numpy as np

docvqa = load_from_disk(os.path.join(cfg.DOCVQA_DIR, 'docvqa_dataset'))
print(f"✓ DocVQA loaded: {len(docvqa['validation'])} val examples")

In [3]:
# ============================================================
# CELL 3: Load Pix2Struct-DocVQA
# ============================================================

from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration
from PIL import Image
from io import BytesIO

print("Loading Pix2Struct-DocVQA model...")
processor = Pix2StructProcessor.from_pretrained(cfg.PIX2STRUCT_ID)
model = Pix2StructForConditionalGeneration.from_pretrained(cfg.PIX2STRUCT_ID)
model = model.to('cuda')
model.eval()

print(f"✓ Model loaded: {cfg.PIX2STRUCT_ID}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# --- Quick test ---
example = docvqa['validation'][0]
img = example['image']
if isinstance(img, dict) and 'bytes' in img:
    img = Image.open(BytesIO(img['bytes']))
img = img.convert('RGB')

inputs = processor(
    images=img,
    text=example['question'],
    return_tensors="pt",
    max_patches=2048
).to('cuda')

with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=256)

prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\nTest example:")
print(f"  Question: {example['question']}")
print(f"  Ground truth: {example['answers']}")
print(f"  Predicted: {prediction}")

Loading Pix2Struct-DocVQA model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

The image processor of type `Pix2StructImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

✓ Model loaded: google/pix2struct-docvqa-base
  Parameters: 282.3M


Arial.TTF: 0.00B [00:00, ?B/s]


Test example:
  Question: what is the page no or page mentioned ?
  Ground truth: ['2', 'page 2']
  Predicted: 2


In [4]:
# ============================================================
# CELL 4: Evaluate Pix2Struct Zero-Shot on DocVQA Validation
# ============================================================

from tqdm import tqdm
import time
import json

def normalized_levenshtein(s1, s2):
    s1, s2 = s1.lower().strip(), s2.lower().strip()
    if s1 == s2:
        return 1.0
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    dist = dp[m][n]
    return 1.0 - (dist / max(m, n))

def compute_anls(prediction, ground_truths, threshold=0.5):
    if not prediction:
        return 0.0
    best = max(normalized_levenshtein(prediction, gt) for gt in ground_truths)
    return best if best >= threshold else 0.0

def compute_exact_match(prediction, ground_truths):
    pred = prediction.lower().strip()
    return float(any(pred == gt.lower().strip() for gt in ground_truths))

def pix2struct_predict(image, question):
    if isinstance(image, dict) and 'bytes' in image:
        image = Image.open(BytesIO(image['bytes']))
    image = image.convert('RGB')

    inputs = processor(
        images=image,
        text=question,
        return_tensors="pt",
        max_patches=2048
    ).to('cuda')

    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=256)

    return processor.decode(generated[0], skip_special_tokens=True)


# --- Run evaluation on same 500 examples as baseline ---
NUM_EVAL = 500
val_data = docvqa['validation']
np.random.seed(42)
eval_indices = np.random.choice(len(val_data), NUM_EVAL, replace=False)

results = []
anls_scores = []
em_scores = []

print(f"Evaluating Pix2Struct zero-shot on {NUM_EVAL} DocVQA validation examples...")
print("=" * 60)

start_time = time.time()

for i, idx in enumerate(tqdm(eval_indices, desc="Evaluating")):
    example = val_data[int(idx)]
    question = example['question']
    ground_truths = example['answers']

    prediction = pix2struct_predict(example['image'], question)

    anls = compute_anls(prediction, ground_truths)
    em = compute_exact_match(prediction, ground_truths)

    anls_scores.append(anls)
    em_scores.append(em)

    results.append({
        'index': int(idx),
        'question': question,
        'ground_truths': ground_truths,
        'prediction': prediction,
        'anls': anls,
        'exact_match': em,
    })

elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"PIX2STRUCT ZERO-SHOT RESULTS ({NUM_EVAL} examples)")
print("=" * 60)
print(f"  ANLS:         {np.mean(anls_scores):.4f}")
print(f"  Exact Match:  {np.mean(em_scores):.4f}")
print(f"  Time:         {elapsed:.0f}s ({elapsed/NUM_EVAL:.1f}s per example)")
print("=" * 60)

# Compare with baseline
print(f"\n  vs Baseline:  ANLS +{np.mean(anls_scores) - 0.326:.4f}")
print(f"                EM   +{np.mean(em_scores) - 0.220:.4f}")

# Save results
results_path = os.path.join(cfg.OUTPUTS_DIR, 'pix2struct_zeroshot_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'model': f'Pix2Struct zero-shot ({cfg.PIX2STRUCT_ID})',
        'dataset': 'DocVQA validation',
        'num_examples': NUM_EVAL,
        'anls': float(np.mean(anls_scores)),
        'exact_match': float(np.mean(em_scores)),
        'per_example': results
    }, f, indent=2)
print(f"\n✓ Results saved to {results_path}")

Evaluating Pix2Struct zero-shot on 500 DocVQA validation examples...


Evaluating: 100%|██████████| 500/500 [09:35<00:00,  1.15s/it]


PIX2STRUCT ZERO-SHOT RESULTS (500 examples)
  ANLS:         0.6548
  Exact Match:  0.5400
  Time:         576s (1.2s per example)

  vs Baseline:  ANLS +0.3288
                EM   +0.3200

✓ Results saved to /content/drive/MyDrive/FinDocVQA/outputs/pix2struct_zeroshot_results.json
